# Validation and Locked-Test Descriptive Comparison

This notebook documents the descriptive comparison of the validation and locked-test partitions after the modelling protocol was fixed. It examines aggregate partition composition, review characteristics, and metadata coverage without tuning a model or selecting a final model.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level audit records, predictions, checkpoints, or intermediate artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the restricted descriptive-analysis contract

Imports the analysis libraries, defines a restricted project-root placeholder, records the validation and locked-test input locations, and specifies the required columns and feature groups. The placeholder avoids exposing a workstation path.


In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
REPORT_DIR = ROUND1_DIR / 'reports' / 'validation_test_shift'
FIGURE_DIR = ROUND1_DIR / 'figures' / 'validation_test_shift'

VALIDATION_PATH = PROJECT_ROOT / 'data' / 'splits' / 'validation.csv'
TEST_PATH = PROJECT_ROOT / 'data' / 'final_test_locked' / 'test_locked.csv'
SPLIT_METADATA_PATH = PROJECT_ROOT / 'data' / 'splits' / 'split_metadata.json'

LABEL_ORDER = ['negatif', 'netral', 'positif']
NUMERIC_FEATURES = [
    'word_count', 'char_len', 'confidence', 'exclamation_count', 'question_count',
]
BINARY_FEATURES = ['has_emoji', 'has_negation', 'has_elongation', 'has_uppercase']
REQUIRED_COLUMNS = {
    'Place', 'Actual Place Name', 'Place Type', 'text_clean', 'sentiment_label',
    *NUMERIC_FEATURES, *BINARY_FEATURES,
}

print(f'Restricted project root placeholder: {PROJECT_ROOT}')
print(f'Reports: {REPORT_DIR}')
print(f'Figures: {FIGURE_DIR}')


### Load and validate the two evaluation partitions

Loads the restricted validation and locked-test files, validates their schema and silver-label values, and checks row counts against split metadata. The locked-test silver labels are used here only for post-protocol descriptive reporting.


In [ ]:
def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'File {split_name} tidak ditemukan: {path}')

    frame = pd.read_csv(path)
    missing_columns = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing_columns:
        raise ValueError(f'{split_name} kehilangan kolom wajib: {missing_columns}')

    frame = frame.copy()
    frame['dataset_split'] = split_name
    frame['sentiment_label'] = frame['sentiment_label'].astype(str).str.strip().str.lower()
    unexpected_labels = sorted(set(frame['sentiment_label']) - set(LABEL_ORDER))
    if unexpected_labels:
        raise ValueError(f'{split_name} memiliki sentiment_label tak dikenal: {unexpected_labels}')

    for column in NUMERIC_FEATURES:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')

    for column in BINARY_FEATURES:
        normalized = frame[column].astype(str).str.strip().str.lower()
        valid_values = {'true', 'false', '1', '0', 'yes', 'no', 'nan', ''}
        invalid_values = sorted(set(normalized) - valid_values)
        if invalid_values:
            raise ValueError(f'{split_name}.{column} memiliki nilai boolean tidak dikenal: {invalid_values[:10]}')
        frame[column] = normalized.map({'true': True, '1': True, 'yes': True, 'false': False, '0': False, 'no': False})

    for column in ['Place', 'Actual Place Name', 'Place Type', 'text_clean']:
        frame[column] = frame[column].fillna('').astype(str).str.strip()

    return frame


validation = load_split(VALIDATION_PATH, 'validation')
locked_test = load_split(TEST_PATH, 'locked_test')

with SPLIT_METADATA_PATH.open(encoding='utf-8') as handle:
    split_metadata = json.load(handle)

expected_validation_n = split_metadata['splits']['validation']['n_samples']
expected_test_n = split_metadata['splits']['test']['n_samples']
assert len(validation) == expected_validation_n, (len(validation), expected_validation_n)
assert len(locked_test) == expected_test_n, (len(locked_test), expected_test_n)

print(f'Validation rows: {len(validation):,}')
print(f'Locked-test rows: {len(locked_test):,}')
display(pd.concat([
    validation['sentiment_label'].value_counts().rename('validation'),
    locked_test['sentiment_label'].value_counts().rename('locked_test'),
], axis=1).reindex(LABEL_ORDER).fillna(0).astype(int))


### Audit partition integrity under the grouping rule

Calculates exact-case Actual Place Name overlap, raw-place overlap, and case-folded clean-text hash overlap. It writes an aggregate integrity table and stops interpretation if the specified business-place grouping rule is violated.


In [ ]:
def nonempty_set(frame: pd.DataFrame, column: str) -> set[str]:
    return set(frame.loc[frame[column].ne(''), column])


validation_groups = nonempty_set(validation, 'Actual Place Name')
test_groups = nonempty_set(locked_test, 'Actual Place Name')
validation_places = nonempty_set(validation, 'Place')
test_places = nonempty_set(locked_test, 'Place')
validation_text_hashes = {
    hashlib.sha256(text.casefold().encode('utf-8')).hexdigest()
    for text in validation.loc[validation['text_clean'].ne(''), 'text_clean']
}
test_text_hashes = {
    hashlib.sha256(text.casefold().encode('utf-8')).hexdigest()
    for text in locked_test.loc[locked_test['text_clean'].ne(''), 'text_clean']
}

integrity_rows = [
    {'check': 'validation_rows_match_metadata', 'value': len(validation), 'expected_or_note': expected_validation_n, 'status': 'pass' if len(validation) == expected_validation_n else 'fail'},
    {'check': 'locked_test_rows_match_metadata', 'value': len(locked_test), 'expected_or_note': expected_test_n, 'status': 'pass' if len(locked_test) == expected_test_n else 'fail'},
    {'check': 'actual_place_name_overlap', 'value': len(validation_groups & test_groups), 'expected_or_note': '0 for business-place group split', 'status': 'pass' if not (validation_groups & test_groups) else 'review'},
    {'check': 'raw_place_overlap', 'value': len(validation_places & test_places), 'expected_or_note': 'reported for context only', 'status': 'pass' if not (validation_places & test_places) else 'review'},
    {'check': 'exact_text_hash_overlap', 'value': len(validation_text_hashes & test_text_hashes), 'expected_or_note': 'reported only; text equality is not the split grouping rule', 'status': 'descriptive'},
]
integrity_checks = pd.DataFrame(integrity_rows)
integrity_checks.to_csv(REPORT_DIR / 'validation_test_integrity_checks.csv', index=False)

if validation_groups & test_groups:
    raise RuntimeError('Actual Place Name overlap ditemukan. Hentikan interpretasi sebelum split diperiksa.')

display(integrity_checks)


### Create aggregate descriptive tables

Builds aggregate tables for partition size, group and place-type coverage, silver-label balance, numeric review features, text-indicator prevalence, and the top Place Type categories. The generated CSV files contain no review-level text in the release whitelist.


In [ ]:
frames = {'validation': validation, 'locked_test': locked_test}

overview_rows = []
for split_name, frame in frames.items():
    overview_rows.append({
        'dataset_split': split_name,
        'n_reviews': len(frame),
        'n_unique_actual_place_name': frame.loc[frame['Actual Place Name'].ne(''), 'Actual Place Name'].nunique(),
        'n_unique_place_values': frame.loc[frame['Place'].ne(''), 'Place'].nunique(),
        'n_unique_place_types': frame.loc[frame['Place Type'].ne(''), 'Place Type'].nunique(),
        'reviews_per_actual_place_mean': len(frame) / max(frame.loc[frame['Actual Place Name'].ne(''), 'Actual Place Name'].nunique(), 1),
        'missing_actual_place_name_n': int(frame['Actual Place Name'].eq('').sum()),
        'missing_place_type_n': int(frame['Place Type'].eq('').sum()),
        'missing_clean_text_n': int(frame['text_clean'].eq('').sum()),
    })
overview = pd.DataFrame(overview_rows)
overview.to_csv(REPORT_DIR / 'validation_test_descriptive_overview.csv', index=False)

label_distribution = (
    pd.concat([
        frame['sentiment_label'].value_counts().rename_axis('silver_label').rename('n').to_frame().assign(dataset_split=split_name)
        for split_name, frame in frames.items()
    ])
    .reset_index()
)
label_distribution['silver_label'] = pd.Categorical(label_distribution['silver_label'], categories=LABEL_ORDER, ordered=True)
label_distribution = label_distribution.sort_values(['silver_label', 'dataset_split']).reset_index(drop=True)
label_distribution['share_pct'] = label_distribution.groupby('dataset_split')['n'].transform(lambda values: values / values.sum() * 100)
label_distribution.to_csv(REPORT_DIR / 'validation_test_silver_label_distribution.csv', index=False)

numeric_rows = []
for split_name, frame in frames.items():
    for feature in NUMERIC_FEATURES:
        values = frame[feature].dropna()
        numeric_rows.append({
            'dataset_split': split_name, 'feature': feature, 'n_nonmissing': len(values),
            'n_missing': int(frame[feature].isna().sum()), 'mean': values.mean(), 'std': values.std(),
            'min': values.min(), 'q25': values.quantile(0.25), 'median': values.median(),
            'q75': values.quantile(0.75), 'p95': values.quantile(0.95), 'max': values.max(),
        })
numeric_summary = pd.DataFrame(numeric_rows)
numeric_summary.to_csv(REPORT_DIR / 'validation_test_numeric_feature_summary.csv', index=False)

by_label_rows = []
for split_name, frame in frames.items():
    for label, label_frame in frame.groupby('sentiment_label', observed=True):
        for feature in ['word_count', 'char_len', 'confidence']:
            values = label_frame[feature].dropna()
            by_label_rows.append({
                'dataset_split': split_name, 'silver_label': label, 'feature': feature,
                'n': len(values), 'mean': values.mean(), 'median': values.median(),
                'q25': values.quantile(0.25), 'q75': values.quantile(0.75),
            })
numeric_by_label = pd.DataFrame(by_label_rows)
numeric_by_label.to_csv(REPORT_DIR / 'validation_test_numeric_features_by_silver_label.csv', index=False)

binary_rows = []
for split_name, frame in frames.items():
    for feature in BINARY_FEATURES:
        known = frame[feature].dropna()
        binary_rows.append({
            'dataset_split': split_name, 'feature': feature, 'n_nonmissing': len(known),
            'n_true': int(known.sum()), 'share_true_pct': float(known.mean() * 100) if len(known) else np.nan,
            'n_missing': int(frame[feature].isna().sum()),
        })
binary_summary = pd.DataFrame(binary_rows)
binary_summary.to_csv(REPORT_DIR / 'validation_test_binary_feature_prevalence.csv', index=False)

type_count_frames = []
for split_name, frame in frames.items():
    split_type_counts = (
        frame.loc[frame['Place Type'].ne(''), 'Place Type']
        .value_counts()
        .rename_axis('place_type')
        .rename('n')
        .reset_index()
    )
    split_type_counts['dataset_split'] = split_name
    type_count_frames.append(split_type_counts)

type_counts = pd.concat(type_count_frames, ignore_index=True)
type_counts['share_pct'] = type_counts.groupby('dataset_split')['n'].transform(lambda values: values / values.sum() * 100)
type_counts.to_csv(REPORT_DIR / 'validation_test_place_type_distribution.csv', index=False)

type_pivot = (
    type_counts.pivot_table(index='place_type', columns='dataset_split', values='n', aggfunc='sum', fill_value=0)
    .reindex(columns=['validation', 'locked_test'], fill_value=0)
    .astype(int)
)
type_pivot['total_n'] = type_pivot.sum(axis=1)
type_pivot['coverage_status'] = np.select(
    [(type_pivot['validation'] > 0) & (type_pivot['locked_test'] > 0),
     type_pivot['validation'] > 0],
    ['shared', 'validation_only'], default='locked_test_only',
)
type_coverage = type_pivot.reset_index().sort_values(['total_n', 'place_type'], ascending=[False, True])
type_coverage.to_csv(REPORT_DIR / 'validation_test_place_type_coverage.csv', index=False)
top_place_types = type_coverage.head(20).copy()
top_place_types.to_csv(REPORT_DIR / 'validation_test_place_type_top20.csv', index=False)

display(overview)
display(label_distribution)
display(numeric_summary.round(4))
display(binary_summary.round(3))
display(top_place_types.head(10))


### Generate diagnostic comparison figures

Creates aggregate figures for silver-label composition, text indicators, review length, LSTM confidence, and frequent Place Type categories. These figure-producing cells are retained because their final outputs support the revised manuscript and supplementary material.


In [ ]:
plt.rcParams.update({'figure.dpi': 140, 'savefig.dpi': 220, 'font.size': 10})
colors = {'validation': '#2a6f97', 'locked_test': '#d1495b'}

label_plot = label_distribution.pivot(index='silver_label', columns='dataset_split', values='share_pct').reindex(LABEL_ORDER)
binary_plot = binary_summary.pivot(index='feature', columns='dataset_split', values='share_true_pct').reindex(BINARY_FEATURES)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), constrained_layout=True)
label_plot.plot(kind='bar', ax=axes[0], color=[colors['validation'], colors['locked_test']], width=0.76)
axes[0].set_title('Silver-label distribution')
axes[0].set_xlabel('Silver label')
axes[0].set_ylabel('Reviews (%)')
axes[0].set_ylim(0, max(70, float(label_plot.max().max()) + 8))
axes[0].legend(title='Split', labels=['Validation', 'Locked test'])
axes[0].tick_params(axis='x', rotation=0)

binary_plot.rename(index=lambda value: value.replace('has_', '').replace('_', ' ')).plot(
    kind='bar', ax=axes[1], color=[colors['validation'], colors['locked_test']], width=0.76
)
axes[1].set_title('Text-indicator prevalence')
axes[1].set_xlabel('Indicator')
axes[1].set_ylabel('Reviews with indicator (%)')
axes[1].legend(title='Split', labels=['Validation', 'Locked test'])
axes[1].tick_params(axis='x', rotation=20)
fig.savefig(FIGURE_DIR / 'validation_test_class_and_binary_features.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5), constrained_layout=True)
for split_name, frame in frames.items():
    word_limit = max(validation['word_count'].quantile(0.99), locked_test['word_count'].quantile(0.99))
    char_limit = max(validation['char_len'].quantile(0.99), locked_test['char_len'].quantile(0.99))
    axes[0, 0].hist(frame['word_count'].clip(upper=word_limit).dropna(), bins=35, density=True, alpha=0.55, label=split_name.replace('_', ' '), color=colors[split_name])
    axes[0, 1].hist(frame['char_len'].clip(upper=char_limit).dropna(), bins=35, density=True, alpha=0.55, label=split_name.replace('_', ' '), color=colors[split_name])
axes[0, 0].set_title('Review word count (clipped at pooled p99)')
axes[0, 0].set_xlabel('Words')
axes[0, 0].set_ylabel('Density')
axes[0, 1].set_title('Review character length (clipped at pooled p99)')
axes[0, 1].set_xlabel('Characters')
axes[0, 1].set_ylabel('Density')

box_data_words = [validation['word_count'].dropna(), locked_test['word_count'].dropna()]
box_data_conf = [validation['confidence'].dropna(), locked_test['confidence'].dropna()]
axes[1, 0].boxplot(box_data_words, tick_labels=['Validation', 'Locked test'], showfliers=False)
axes[1, 0].set_title('Review word-count spread')
axes[1, 0].set_ylabel('Words')
axes[1, 1].boxplot(box_data_conf, tick_labels=['Validation', 'Locked test'], showfliers=False)
axes[1, 1].set_title('LSTM silver-label confidence spread')
axes[1, 1].set_ylabel('Confidence')
for axis in axes.flat[:2]:
    axis.legend(title='Split')
fig.savefig(FIGURE_DIR / 'validation_test_text_length_and_confidence.png', bbox_inches='tight')
plt.show()

top_plot = (
    top_place_types
    .sort_values('total_n', ascending=True)
    .set_index('place_type')
    .reindex(columns=['validation', 'locked_test'], fill_value=0)
)
fig_height = max(5.5, 0.34 * len(top_plot) + 1.8)
fig, ax = plt.subplots(figsize=(10.5, fig_height), constrained_layout=True)
top_plot.plot(kind='barh', ax=ax, color=[colors['validation'], colors['locked_test']], width=0.78)
ax.set_title('Most frequent Place Type categories across validation and locked test')
ax.set_xlabel('Reviews (count)')
ax.set_ylabel('Place Type')
ax.legend(title='Split', labels=['Validation', 'Locked test'])
fig.savefig(FIGURE_DIR / 'validation_test_place_type_top20.png', bbox_inches='tight')
plt.show()


### Record the internal descriptive-analysis manifest

Writes the authorized-environment manifest with input checksums, feature definitions, and generated artifact paths. The raw internal manifest is not a public artifact because it can reference restricted inputs.


In [ ]:
output_files = sorted(
    [str(path.relative_to(ROUND1_DIR)) for path in REPORT_DIR.glob('*')]
    + [str(path.relative_to(ROUND1_DIR)) for path in FIGURE_DIR.glob('*')]
)
manifest = {
    'analysis': 'R1_05 validation-test descriptive shift',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'purpose': 'Descriptive comparison of validation and locked-test data; no model tuning or hypothesis testing.',
    'input_files': {
        'validation': {'path': str(VALIDATION_PATH.relative_to(PROJECT_ROOT)), 'sha256': sha256_file(VALIDATION_PATH), 'n_rows': len(validation)},
        'locked_test': {'path': str(TEST_PATH.relative_to(PROJECT_ROOT)), 'sha256': sha256_file(TEST_PATH), 'n_rows': len(locked_test)},
        'split_metadata': {'path': str(SPLIT_METADATA_PATH.relative_to(PROJECT_ROOT)), 'sha256': sha256_file(SPLIT_METADATA_PATH)},
    },
    'grouping_rule': 'Actual Place Name',
    'group_overlap_n': int(len(validation_groups & test_groups)),
    'exact_clean_text_hash_overlap_n': int(len(validation_text_hashes & test_text_hashes)),
    'silver_label_order': LABEL_ORDER,
    'numeric_features': NUMERIC_FEATURES,
    'binary_features': BINARY_FEATURES,
    'outputs': output_files,
    'environment': {
        'python': sys.version, 'platform': platform.platform(),
        'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
    },
}
with (REPORT_DIR / 'validation_test_descriptive_shift_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

print('R1_05 selesai. Output agregat:')
for output_file in output_files:
    print(f'- {output_file}')

print('\nInterpretation guardrail: gunakan hasil ini untuk konteks distribusi split, bukan sebagai bukti kausal atau dasar pemilihan model.')


### Create English-labelled publication figures

Applies display-only English labels to the aggregate plots and assembles Supplementary Figure S4. The translation mapping changes neither source metadata nor model inputs, and the final figure is distributed separately through the aggregate-output whitelist.


In [ ]:
import matplotlib.image as mpimg

DISPLAY_CLASS_LABELS = {'negatif': 'negative', 'netral': 'neutral', 'positif': 'positive'}
DISPLAY_BINARY_LABELS = {
    'has_emoji': 'emoji', 'has_negation': 'negation',
    'has_elongation': 'elongation', 'has_uppercase': 'uppercase',
}
DISPLAY_PLACE_TYPE_LABELS = {
    'Restoran': 'Restaurant',
    'Kantor Perusahaan': 'Corporate Office',
    'Toko': 'Store',
    'Kedai Kopi': 'Coffee Shop',
    'Masjid': 'Mosque',
    'Komplek perumahan': 'Residential Complex',
    'Kafe': 'Cafe',
    'Minimarket': 'Convenience Store',
    'Jenis tempat tidak ditemukan': 'Unspecified place type',
    'Bengkel Sepeda Motor': 'Motorcycle Repair Shop',
    'Toko bahan makanan': 'Grocery Store',
    'Bengkel Mobil': 'Auto Repair Shop',
    'Perumahan': 'Housing Development',
    'Toko Bahan Bangunan': 'Building Materials Store',
    'Kedai Sarapan & Makan Siang': 'Breakfast and Lunch Restaurant',
    'Kantor Pemerintah': 'Government Office',
    'Toko Pakaian': 'Clothing Store',
    'Tempat Cukur Rambut': 'Barbershop',
    'Pusat Perbelanjaan': 'Shopping Center',
    'Toko Elektronik': 'Electronics Store',
}

label_plot_display = label_plot.rename(index=DISPLAY_CLASS_LABELS).reindex(['negative', 'neutral', 'positive'])
binary_plot_display = binary_plot.rename(index=DISPLAY_BINARY_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), constrained_layout=True)
label_plot_display.plot(kind='bar', ax=axes[0], color=[colors['validation'], colors['locked_test']], width=0.76)
axes[0].set_title('Silver-label distribution')
axes[0].set_xlabel('Silver label')
axes[0].set_ylabel('Reviews (%)')
axes[0].set_ylim(0, max(70, float(label_plot_display.max().max()) + 8))
axes[0].legend(title='Split', labels=['Validation', 'Locked test'])
axes[0].tick_params(axis='x', rotation=0)
binary_plot_display.plot(kind='bar', ax=axes[1], color=[colors['validation'], colors['locked_test']], width=0.76)
axes[1].set_title('Text-indicator prevalence')
axes[1].set_xlabel('Indicator')
axes[1].set_ylabel('Reviews with indicator (%)')
axes[1].legend(title='Split', labels=['Validation', 'Locked test'])
axes[1].tick_params(axis='x', rotation=20)
class_binary_figure = FIGURE_DIR / 'validation_test_class_and_binary_features.png'
fig.savefig(class_binary_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5), constrained_layout=True)
word_limit = max(validation['word_count'].quantile(0.99), locked_test['word_count'].quantile(0.99))
char_limit = max(validation['char_len'].quantile(0.99), locked_test['char_len'].quantile(0.99))
for split_name, frame in frames.items():
    display_split = 'Validation' if split_name == 'validation' else 'Locked test'
    axes[0, 0].hist(frame['word_count'].clip(upper=word_limit).dropna(), bins=35, density=True, alpha=0.55, label=display_split, color=colors[split_name])
    axes[0, 1].hist(frame['char_len'].clip(upper=char_limit).dropna(), bins=35, density=True, alpha=0.55, label=display_split, color=colors[split_name])
axes[0, 0].set_title('Review word count (clipped at pooled p99)')
axes[0, 0].set_xlabel('Words')
axes[0, 0].set_ylabel('Density')
axes[0, 1].set_title('Review character length (clipped at pooled p99)')
axes[0, 1].set_xlabel('Characters')
axes[0, 1].set_ylabel('Density')
axes[1, 0].boxplot([validation['word_count'].dropna(), locked_test['word_count'].dropna()], tick_labels=['Validation', 'Locked test'], showfliers=False)
axes[1, 0].set_title('Review word-count spread')
axes[1, 0].set_ylabel('Words')
axes[1, 1].boxplot([validation['confidence'].dropna(), locked_test['confidence'].dropna()], tick_labels=['Validation', 'Locked test'], showfliers=False)
axes[1, 1].set_title('LSTM silver-label confidence spread')
axes[1, 1].set_ylabel('Confidence')
for axis in axes.flat[:2]:
    axis.legend(title='Split')
text_length_figure = FIGURE_DIR / 'validation_test_text_length_and_confidence.png'
fig.savefig(text_length_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

top_plot_display = top_plot.rename(index=lambda value: DISPLAY_PLACE_TYPE_LABELS.get(value, value))
fig_height = max(5.5, 0.34 * len(top_plot_display) + 1.8)
fig, ax = plt.subplots(figsize=(10.5, fig_height), constrained_layout=True)
top_plot_display.plot(kind='barh', ax=ax, color=[colors['validation'], colors['locked_test']], width=0.78)
ax.set_title('Most frequent Place Type categories across validation and locked test')
ax.set_xlabel('Reviews (count)')
ax.set_ylabel('Place Type')
ax.legend(title='Split', labels=['Validation', 'Locked test'])
place_type_figure = FIGURE_DIR / 'validation_test_place_type_top20.png'
fig.savefig(place_type_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(3, 1, figsize=(13.5, 22.0))
for axis, panel, title, image_path in [
    (axes[0], 'A', 'Silver-label distribution and text indicators', class_binary_figure),
    (axes[1], 'B', 'Review length and LSTM confidence', text_length_figure),
    (axes[2], 'C', 'Top-20 Place Type categories', place_type_figure),
]:
    axis.imshow(mpimg.imread(image_path))
    axis.axis('off')
    axis.text(0.01, 0.985, panel, transform=axis.transAxes, ha='left', va='top', fontsize=15, fontweight='bold', color='black', bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.88, 'pad': 2})
    axis.set_title(title, loc='left', fontsize=13, pad=8)
fig.subplots_adjust(left=0.02, right=0.98, bottom=0.015, top=0.985, hspace=0.08)
supplementary_s4_figure = FIGURE_DIR / 'supplementary_figure_s4_validation_test_descriptive_shift.png'
fig.savefig(supplementary_s4_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

manifest_path = REPORT_DIR / 'validation_test_descriptive_shift_manifest.json'
publication_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
publication_manifest['publication_visualization'] = {
    'class_display_labels': DISPLAY_CLASS_LABELS,
    'place_type_display_mapping': DISPLAY_PLACE_TYPE_LABELS,
    'unavailable_metadata_display_label': 'Unspecified place type',
    'display_only_note': 'English display labels do not alter source metadata, split files, or model inputs.',
    'supplementary_figure_s4': str(supplementary_s4_figure.relative_to(PROJECT_ROOT)),
}
publication_manifest['publication_figure_paths'] = [
    str(class_binary_figure.relative_to(PROJECT_ROOT)),
    str(text_length_figure.relative_to(PROJECT_ROOT)),
    str(place_type_figure.relative_to(PROJECT_ROOT)),
    str(supplementary_s4_figure.relative_to(PROJECT_ROOT)),
]
with manifest_path.open('w', encoding='utf-8') as handle:
    json.dump(publication_manifest, handle, ensure_ascii=False, indent=2)
print('Supplementary Figure S4:', supplementary_s4_figure)
